# First we start by downloading all necessary libraries/classes

In [ ]:
!pip install torchinfo

In [ ]:
import matplotlib.pyplot as plt
import zipfile
import torch
import torchvision
from torch import nn
from torchvision import transforms
from torchinfo import summary
import os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from typing import Dict, List, Tuple
from PIL import Image
import requests
import timm
from timm.data import create_transform


In [ ]:
def set_seeds(seed: int=42):
    # Set the seed for general torch operations
    torch.manual_seed(seed)
    # Set the seed for CUDA torch operations (ones that happen on the GPU)
    torch.cuda.manual_seed(seed)

# this code is to check if our code is running on the GPU correctly

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

# Setting up the pretrained weights, defining our classes, and adding a trainable layer

In [ ]:
# Load pretrained  model
pretrained_mobilevit = timm.create_model("mobilevit_s.cvnets_in1k", pretrained=True).to(device)

# Freeze model parameters
for parameter in pretrained_mobilevit.parameters():
    parameter.requires_grad = False

for name, param in list(pretrained_mobilevit.named_parameters())[-10:]:
    param.requires_grad = True
# Define class names
class_names = ['Monkeypox', 'Others']

# Set random seeds (assuming set_seeds is defined)
set_seeds()

# Replace classification head

pretrained_mobilevit.head = nn.Sequential(
    nn.AdaptiveAvgPool2d(1),# Pool to (batch_size, 640, 1, 1)
    nn.Flatten(),
    nn.Linear(640, 256),
    nn.ReLU(),
    nn.Dropout(p=0.3),
    nn.Linear(256, 2)
).to(device)




# Get pretrained config
pretrained_cfg = pretrained_mobilevit.pretrained_cfg

# Extract transform parameters safely
input_size = pretrained_cfg.input_size if hasattr(pretrained_cfg, "input_size") else 256
mean = pretrained_cfg.mean if hasattr(pretrained_cfg, "mean") else (0.5, 0.5, 0.5)
std = pretrained_cfg.std if hasattr(pretrained_cfg, "std") else (0.5, 0.5, 0.5)
interpolation = pretrained_cfg.interpolation if hasattr(pretrained_cfg, "interpolation") else "bilinear"

# Define transforms using timm's recommended settings
pretrained_mobilevit_transforms = create_transform(
    input_size=input_size,
    mean=mean,
    std=std,
    interpolation=interpolation
)

# Print model architecture
#print(pretrained_mobilevit)

# Here is the model's architecture:

In [ ]:
# Print a summary using torchinfo (uncomment for actual output)
summary(model= pretrained_mobilevit ,
        input_size=(16, 3, 256, 256), # (batch_size, color_channels, height, width)
         #col_names=["input_size"], # uncomment for smaller output
         col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"]
)

In [ ]:
model = pretrained_mobilevit  # Make sure the model is assigned to a variable


In [ ]:
for name, param in model.named_parameters():
    print(f"{name}: {'Trainable' if param.requires_grad else 'Frozen'}")


In [ ]:
!unzip /content/FBinaryMSLD.zip -d dataset

In [ ]:
# Setup directory paths to train and test images
train_dir = '/content/dataset/Train'
test_dir = '/content/dataset/Test'

## And now we've got transforms ready, we can turn our images into DataLoaders using the create_dataloaders()

In [ ]:
NUM_WORKERS = os.cpu_count()

def create_dataloaders(
    train_dir: str,
    test_dir: str,
    transform: transforms.Compose,
    batch_size: int,
    num_workers: int=NUM_WORKERS
):

  # Use ImageFolder to create dataset(s)
  train_data = datasets.ImageFolder(train_dir, transform=transform)
  test_data = datasets.ImageFolder(test_dir, transform=transform)

  # Get class names
  class_names = train_data.classes

  # Turn images into data loaders
  train_dataloader = DataLoader(
      train_data,
      batch_size=batch_size,
      shuffle=True,
      num_workers=num_workers,
      pin_memory=True,
  )
  test_dataloader = DataLoader(
      test_data,
      batch_size=batch_size,
      shuffle=False,
      num_workers=num_workers,
      pin_memory=True,
  )

  return train_dataloader, test_dataloader, class_names

In [ ]:
# Setup dataloaders
train_dataloader_pretrained, test_dataloader_pretrained, class_names = create_dataloaders(train_dir=train_dir,
                                                                                                     test_dir=test_dir,
                                                                                                     transform=pretrained_mobilevit_transforms,
                                                                                                     batch_size=16)


# Setting up the training and testing functions and start the training


In [ ]:
from typing import Tuple
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

# Define the training and testing steps with required metrics
def test_step(model: torch.nn.Module,
              dataloader: torch.utils.data.DataLoader,
              loss_fn: torch.nn.Module,
              device: torch.device) -> Tuple[float, float, float, float, float, float]:
    model.eval()
    test_loss, total_correct, total_samples = 0, 0, 0
    all_preds, all_labels = [], []

    with torch.inference_mode():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            test_pred_logits = model(X)

            loss = loss_fn(test_pred_logits, y)
            test_loss += loss.item()

            test_pred_labels = test_pred_logits.argmax(dim=1)
            total_correct += (test_pred_labels == y).sum().item()
            total_samples += y.size(0)

            # Store the predictions and true labels for evaluation
            all_preds.extend(test_pred_labels.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    # Calculate all required metrics
    test_precision = precision_score(all_labels, all_preds, average='weighted')
    test_recall = recall_score(all_labels, all_preds, average='weighted')
    test_f1 = f1_score(all_labels, all_preds, average='weighted')
    test_auc = roc_auc_score(all_labels, all_preds, average='weighted') if len(set(all_labels)) > 2 else roc_auc_score(all_labels, all_preds)


    test_loss = test_loss / len(dataloader)
    test_acc = total_correct / total_samples

    return test_loss, test_acc, test_precision, test_recall, test_f1, test_auc


def train_step(model: torch.nn.Module,
               dataloader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               optimizer: torch.optim.Optimizer,
               device: torch.device) -> Tuple[float, float, float, float, float, float]:
    model.train()
    train_loss, total_correct, total_samples = 0, 0, 0
    all_preds, all_labels = [], []

    for X, y in dataloader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()

        y_pred = model(X)
        loss = loss_fn(y_pred, y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        total_correct += (y_pred.argmax(dim=1) == y).sum().item()
        total_samples += y.size(0)

        # Store the predictions and true labels for evaluation
        all_preds.extend(y_pred.argmax(dim=1).cpu().numpy())
        all_labels.extend(y.cpu().numpy())

    # Calculate all required metrics
    train_precision = precision_score(all_labels, all_preds, average='weighted')
    train_recall = recall_score(all_labels, all_preds, average='weighted')
    train_f1 = f1_score(all_labels, all_preds, average='weighted')
    train_auc = roc_auc_score(all_labels, all_preds, average='weighted') if len(set(all_labels)) > 2 else roc_auc_score(all_labels, all_preds)


    train_loss = train_loss / len(dataloader)
    train_acc = total_correct / total_samples

    return train_loss, train_acc, train_precision, train_recall, train_f1, train_auc

from typing import Tuple
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
import torch
# Define the train function that runs training and evaluation for multiple epochs
def train(model: torch.nn.Module,
          train_dataloader: torch.utils.data.DataLoader,
          test_dataloader: torch.utils.data.DataLoader,
          optimizer: torch.optim.Optimizer,
          loss_fn: torch.nn.Module,
          epochs: int,
          device: torch.device,
          patience: int = 5) -> dict:
    results = {
        "train_loss": [],
        "train_acc": [],
        "train_precision": [],
        "train_recall": [],
        "train_f1": [],
        "train_auc": [],
        "test_loss": [],
        "test_acc": [],
        "test_precision": [],
        "test_recall": [],
        "test_f1": [],
        "test_auc": []
    }

    model.to(device)
    best_loss = float('inf')
    patience_counter = 0

    for epoch in range(epochs):
        # Training step
        train_loss, train_acc, train_precision, train_recall, train_f1, train_auc = train_step(
            model=model,
            dataloader=train_dataloader,
            loss_fn=loss_fn,
            optimizer=optimizer,
            device=device
        )

        # Testing step
        test_loss, test_acc, test_precision, test_recall, test_f1, test_auc = test_step(
            model=model,
            dataloader=test_dataloader,
            loss_fn=loss_fn,
            device=device
        )

        # Print the results for the current epoch
        print(
            f"Epoch: {epoch+1} | "
            f"train_loss: {train_loss:.4f} | "
            f"train_acc: {train_acc:.4f} | "
            f"train_precision: {train_precision:.4f} | "
            f"train_recall: {train_recall:.4f} | "
            f"train_f1: {train_f1:.4f} | "
            f"train_auc: {train_auc:.4f} | "
            f"test_loss: {test_loss:.4f} | "
            f"test_acc: {test_acc:.4f} | "
            f"test_precision: {test_precision:.4f} | "
            f"test_recall: {test_recall:.4f} | "
            f"test_f1: {test_f1:.4f} | "
            f"test_auc: {test_auc:.4f}"
        )

        # Save the results for plotting later
        results["train_loss"].append(train_loss)
        results["train_acc"].append(train_acc)
        results["train_precision"].append(train_precision)
        results["train_recall"].append(train_recall)
        results["train_f1"].append(train_f1)
        results["train_auc"].append(train_auc)
        results["test_loss"].append(test_loss)
        results["test_acc"].append(test_acc)
        results["test_precision"].append(test_precision)
        results["test_recall"].append(test_recall)
        results["test_f1"].append(test_f1)
        results["test_auc"].append(test_auc)

        # Early stopping check
        if test_loss < best_loss:
            best_loss = test_loss
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping triggered at epoch {epoch+1}.")
                break

    return results

# Create optimizer and loss function
optimizer = torch.optim.Adam(params=pretrained_mobilevit.parameters(),
                             lr=1e-4,
                             weight_decay=1e-4)
loss_fn = torch.nn.CrossEntropyLoss()

# Train the classifier head of the pretrained  feature extractor model
set_seeds()

import time

# Measure training time
start_time = time.time()

# Train and test the model, and collect results (including precision, recall, F1, AUC, accuracy)
pretrained_mobilevit_results = train(
    model=pretrained_mobilevit,
    train_dataloader=train_dataloader_pretrained,
    test_dataloader=test_dataloader_pretrained,
    optimizer=optimizer,
    loss_fn=loss_fn,
    epochs=5,
    device=device,
    patience=3 # Stop if no improvement
)

# End timer
end_time = time.time()
total_training_time = end_time - start_time
print(f"\nTotal Training Time: {total_training_time:.2f} seconds")
print(f"Average Training Time per Epoch: {total_training_time/5:.2f} seconds")



#Here is the function to calculate the inference time

In [ ]:
def predict_with_timing(model, image_path, transform, class_names, device):
    model.eval()

    # Load and preprocess image
    image = Image.open(image_path).convert("RGB")
    input_tensor = transform(image).unsqueeze(0).to(device)

    # Warm-up pass (CUDA needs it for more accurate timing)
    with torch.inference_mode():
        _ = model(input_tensor)

    # Measure inference time
    start_time = time.time()
    with torch.inference_mode():
        outputs = model(input_tensor)
    end_time = time.time()

    # Process prediction
    probs = torch.softmax(outputs, dim=1).cpu().numpy()[0]
    pred_class = class_names[probs.argmax()]
    inference_time = (end_time - start_time) * 1000  # in milliseconds

    print(f"Predicted class: {pred_class}")
    print(f"\nClass probabilities: {dict(zip(class_names, probs.round(3)))}")
    print(f"\nInference time: {inference_time:.2f} ms")

    return pred_class, probs, inference_time


In [ ]:
image_path = "/content/dataset/Test/Monkeypox/M40_03.jpg"  #Test the inference time on a random image
predict_with_timing(model, image_path, pretrained_mobilevit_transforms, class_names, device)


# Here we plot our evaluation matrices for the test and train data

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc
import numpy as np
def get_predictions_and_labels(model, dataloader, device):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []

    with torch.inference_mode():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            logits = model(X)
            probs = torch.softmax(logits, dim=1)[:, 1]  # probability of class 1
            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    return np.array(all_preds), np.array(all_labels), np.array(all_probs)

# Get preds, labels, and probs for ROC
y_pred, y_true, y_probs = get_predictions_and_labels(model, test_dataloader_pretrained, device)


In [ ]:
import matplotlib.pyplot as plt
# Define the function to plot the curves for all metrics
def plot_all_metrics(results, y_true, y_pred, y_probs, class_names):
    epochs = range(len(results["train_loss"]))

    plt.figure(figsize=(18, 12))


    # Loss
    plt.subplot(3, 3, 1)
    plt.plot(epochs, results["train_loss"], label="train_loss")
    plt.plot(epochs, results["test_loss"], label="test_loss")
    plt.title("Loss")
    plt.xlabel("Epochs")
    plt.legend()

    # Accuracy
    plt.subplot(3, 3, 2)
    plt.plot(epochs, results["train_acc"], label="train_acc")
    plt.plot(epochs, results["test_acc"], label="test_acc")
    plt.title("Accuracy")
    plt.xlabel("Epochs")
    plt.legend()

    # Recall
    plt.subplot(3, 3, 3)
    plt.plot(epochs, results["train_recall"], label="train_recall")
    plt.plot(epochs, results["test_recall"], label="test_recall")
    plt.title("Recall")
    plt.xlabel("Epochs")
    plt.legend()

    # Precision
    plt.subplot(3, 3, 4)
    plt.plot(epochs, results["train_precision"], label="train_precision")
    plt.plot(epochs, results["test_precision"], label="test_precision")
    plt.title("Precision")
    plt.xlabel("Epochs")
    plt.legend()

    # F1 Score
    plt.subplot(3, 3, 5)
    plt.plot(epochs, results["train_f1"], label="train_f1")
    plt.plot(epochs, results["test_f1"], label="test_f1")
    plt.title("F1 Score")
    plt.xlabel("Epochs")
    plt.legend()

    # AUC
    plt.subplot(3, 3, 6)
    plt.plot(epochs, results["train_auc"], label="train_auc")
    plt.plot(epochs, results["test_auc"], label="test_auc")
    plt.title("AUC")
    plt.xlabel("Epochs")
    plt.legend()

    # Confusion Matrix
    plt.subplot(3, 3, 7)
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(cmap=plt.cm.Blues, ax=plt.gca(), colorbar=False)
    plt.title("Confusion Matrix")

    # ROC Curve
    plt.subplot(3, 3, 8)
    fpr, tpr, _ = roc_curve(y_true, y_probs)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"ROC Curve (AUC = {roc_auc:.2f})")
    plt.plot([0, 1], [0, 1], "k--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve")
    plt.legend(loc="lower right")

    # Show the plots
    plt.tight_layout()
    plt.show()



In [ ]:
plot_all_metrics(pretrained_mobilevit_results, y_true, y_pred, y_probs, class_names)


# Let's make Prediction:

In [ ]:
def pred_and_plot_image(
    model: torch.nn.Module,
    class_names: List[str],
    image_path: str,
    image_size: Tuple[int, int] = (224, 224),
    transform: torchvision.transforms = None,
    device: torch.device = device,
):

    # Open image
    img = Image.open(image_path)

    # Create transformation for image (if one doesn't exist)
    if transform is not None:
        image_transform = transform
    else:
        image_transform = transforms.Compose(
            [
                transforms.Resize(image_size),
                transforms.ToTensor(),
                transforms.Normalize(
                    mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
                ),
            ]
        )

    ### Predict on image ###

    # Make sure the model is on the target device
    model.to(device)

    # Turn on model evaluation mode and inference mode
    model.eval()
    with torch.inference_mode():
        # Transform and add an extra dimension to image (model requires samples in [batch_size, color_channels, height, width])
        transformed_image = image_transform(img).unsqueeze(dim=0)

        # Make a prediction on image with an extra dimension and send it to the target device
        target_image_pred = model(transformed_image.to(device))

    # Convert logits -> prediction probabilities (using torch.softmax() for multi-class classification)
    target_image_pred_probs = torch.softmax(target_image_pred, dim=1)

    # Convert prediction probabilities -> prediction labels
    target_image_pred_label = torch.argmax(target_image_pred_probs, dim=1)

    # Plot image with predicted label and probability
    plt.figure()
    plt.imshow(img)
    plt.title(
        f"Pred: {class_names[target_image_pred_label]} | Prob: {target_image_pred_probs.max():.3f}"
    )
    plt.axis(False)


In [ ]:
# Setup custom image path
custom_image_path = "/content/dataset/Test/Others/NM41_01.jpg"
# Predict on custom image
pred_and_plot_image(model=pretrained_mobilevit,
                    image_path=custom_image_path,
                    class_names=class_names)

In [ ]:
# Setup custom image path
custom_image_path = "/content/dataset/Test/Monkeypox/M51_02.jpg"

# Predict on custom image
pred_and_plot_image(model=pretrained_mobilevit,
                    image_path=custom_image_path,
                    class_names=class_names)